# Segmentation Debug

Step through every segmentation stage on a synthetic document image to verify that line / word / character segmenters work correctly.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from src.segmentation.line_segmenter import segment_lines
from src.segmentation.word_segmenter import segment_words
from src.segmentation.char_segmenter import segment_chars
from src.visualization import draw_bboxes  # optional helper

## 1. Synthetic document

In [ ]:
# White page with three rows of black rectangles simulating text lines
doc = np.ones((200, 400), dtype=np.uint8) * 255

# Row 0
for x in [10, 50, 90, 130]:
    doc[20:40, x:x+30] = 0

# Row 1
for x in [10, 60, 110]:
    doc[70:90, x:x+40] = 0

# Row 2
for x in [10, 80]:
    doc[130:150, x:x+60] = 0

plt.figure(figsize=(8, 4))
plt.imshow(doc, cmap='gray')
plt.title('Synthetic document')
plt.axis('off')
plt.show()

## 2. Line segmentation

In [ ]:
lines = segment_lines(doc)
print(f'Detected {len(lines)} lines')

fig, axes = plt.subplots(1, len(lines), figsize=(12, 3))
if len(lines) == 1:
    axes = [axes]
for ax, (line_img, bbox) in zip(axes, lines):
    ax.imshow(line_img, cmap='gray')
    ax.set_title(f'y={bbox[1]}')
    ax.axis('off')
plt.suptitle('Segmented lines')
plt.tight_layout()
plt.show()

## 3. Word segmentation on first line

In [ ]:
if lines:
    first_line_img, _ = lines[0]
    words = segment_words(first_line_img)
    print(f'  Line 0 → {len(words)} words')

    fig, axes = plt.subplots(1, max(len(words), 1), figsize=(10, 3))
    if len(words) == 1:
        axes = [axes]
    for ax, (word_img, _) in zip(axes, words):
        ax.imshow(word_img, cmap='gray')
        ax.axis('off')
    plt.suptitle('Words in first line')
    plt.tight_layout()
    plt.show()

## 4. Character segmentation on first word

In [ ]:
if lines and words:
    first_word_img, _ = words[0]
    chars = segment_chars(first_word_img)
    print(f'  Word 0 → {len(chars)} chars')

    if chars:
        fig, axes = plt.subplots(1, len(chars), figsize=(8, 3))
        if len(chars) == 1:
            axes = [axes]
        for ax, (ch_img, _) in zip(axes, chars):
            ax.imshow(ch_img, cmap='gray')
            ax.axis('off')
        plt.suptitle('Characters in first word')
        plt.tight_layout()
        plt.show()

## 5. Projection profiles

In [ ]:
# Horizontal projection (row pixel sums) — useful for tuning line_gap_threshold
ink = (255 - doc)  # invert so ink = high values
hproj = ink.sum(axis=1)

plt.figure(figsize=(6, 4))
plt.plot(hproj)
plt.title('Horizontal projection profile')
plt.xlabel('Row index')
plt.ylabel('Ink sum')
plt.tight_layout()
plt.show()